In [1]:
import os
import requests
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
Weather_API_Key = os.getenv("WEATHER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


In [2]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="openai/gpt-oss-120B", model_provider="groq")

In [4]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x0000029325F20050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029325F20D70>, model_name='openai/gpt-oss-120B', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
# Create a weather tool
from langchain.tools import tool
url = "https://api.openweathermap.org/data/2.5/weather?"
@tool
def getweather(location:str)->str:
    """Take the location and return the current weather. """
    params={
        "q": location, 
        "appid" : Weather_API_Key, 
        "units" : "imperial"
    }

    try: 
        response = requests.get(url, params=params, timeout = 10)
        return response.json()
    except:
        return f"The requested weather information is unavailable."

In [6]:
# Search tools
from langchain_tavily import TavilySearch
web_search = TavilySearch(
    max_results=3, 
    topic="general", 
    search_depth="basic"
)

In [32]:
from langchain_community.tools import DuckDuckGoSearchRun

duck_search = DuckDuckGoSearchRun()

C:\Users\ramma\AppData\Local\Temp\ipykernel_5412\169271857.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
prompt = """
    You are a helpful chatbot assistant. Use getweather tool for answering weather related queries. Use web_search tool for recent events and the latest, including for finance, news, pop culture and internet search. Use web_search for any queries involving today or the current year (2026). Use duck_search when web_search is not available or returns no results. For normal chat conversation, don't use a tool, just answer on your own. 
"""

In [34]:
from langchain.agents import create_agent
tools = [getweather, duck_search]
agent = create_agent(
    model = llm, 
    tools = tools, 
    system_prompt=prompt
)

In [35]:
# test the agent
from langchain.messages import SystemMessage, HumanMessage, AIMessage
messages=[]

In [36]:
user_query="What is the weather in Peoria today and who won world cup 2026"

In [37]:
messages.append(HumanMessage(content=user_query))

In [38]:
result = agent.invoke({"messages":messages})

In [39]:
print(result["messages"][-1].content)

**Weather in Peoria, Illinois (today)**  
- **Condition:** Overcast clouds (100 % cloud cover)  
- **Temperature:** 75.96 °F (≈ 24.4 °C) – feels like 76.91 °F  
- **Humidity:** 78 %  
- **Wind:** 5.75 mph from the west‑northwest (290°)  
- **Pressure:** 1017 hPa  
- **Visibility:** 10 km (clear visibility despite clouds)  

So it’s a warm, overcast day with mild winds.

**2026 FIFA World Cup winner**  
- **Champion:** **Spain**  
- **Final score:** Spain 1 – 0 Argentina (after extra time)  
- **Venue:** MetLife Stadium (New York‑New Jersey area)  
- **Key moment:** Ferran Torres scored the winning goal in extra time, giving Spain their second World Cup title.


In [40]:
for message in result["messages"]:
    print(message)

content='What is the weather in Peoria today and who won world cup 2026' additional_kwargs={} response_metadata={} id='bbe806a3-290a-404c-a4ca-7101eb6c5071'
content='' additional_kwargs={'reasoning_content': 'We need to answer two parts: weather in Peoria today, and who won World Cup 2026. Use getweather for weather, and web_search for world cup winner (but web_search tool not listed; only duckduckgo_search is available). The developer says: Use web_search for recent events, finance, news, pop culture and internet search. Use duck_search when web_search is not available or returns no results. We have only duckduckgo_search. So we should use duckduckgo_search for world cup winner.\n\nWe need to call getweather with location "Peoria". Probably need to specify city and maybe state? There is Peoria, Illinois. Use location "Peoria, IL". Then duckduckgo_search query "2026 FIFA World Cup winner". Let\'s do calls.', 'tool_calls': [{'id': 'fc_4d537bed-19ae-4536-ba25-fbe9c6281096', 'function': {

In [41]:
# steraming the output

messages

[HumanMessage(content='What is the weather in Peoria today and who won world cup 2026', additional_kwargs={}, response_metadata={}, id='bbe806a3-290a-404c-a4ca-7101eb6c5071')]

In [42]:
for chunk in agent.stream({"messages":messages}, stream_mode = "updates", version="v2"):
    print(chunk['data'])

{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer two parts: weather in Peoria today, and who won World Cup 2026. Both require current info. Use getweather for weather, and web_search (or duckduckgo) for World Cup winner. There\'s no web_search tool defined; only duckduckgo_search. So use getweather for Peoria, and duckduckgo_search for winner. Let\'s call getweather with location "Peoria". Probably need to specify city state? Could be Peoria, Illinois. Use "Peoria, IL". Then search query "who won the 2026 FIFA World Cup".', 'tool_calls': [{'id': 'fc_c4b28dcb-0ac4-479c-bb9c-426977159d8c', 'function': {'arguments': '{"location":"Peoria, IL"}', 'name': 'getweather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 274, 'total_tokens': 426, 'completion_time': 0.31832937, 'completion_tokens_details': {'reasoning_tokens': 122}, 'prompt_time': 0.013291846, 'prompt_tokens_details': None

In [43]:
for chunk in agent.stream({"messages":messages}, stream_mode = "updates", version="v2"):
    if(chunk["type"]=="message"):
        token, mmetadate = chunk["data"]
        print(token)

In [44]:
final_result = chunk['data']['model']["messages"][0].content
final_result

'**Weather in Peoria, Illinois (today)**  \n- **Condition:** Overcast clouds (100\u202f% cloud cover)  \n- **Temperature:** About **76\u202f°F** (feels like 77\u202f°F)  \n- **Humidity:** **78\u202f%**  \n- **Wind:** Around **5.8\u202fmph** from the west‑northwest (290°)  \n- **Visibility:** 10\u202fkm (clear of fog)  \n\n**2026 FIFA World Cup winner**  \n- **Spain** won the 2026 World Cup, defeating **Argentina 1‑0** after extra time in the final held at MetLife Stadium (East Rutherford, New\u202fJersey). Ferran\u202fTorres scored the winning goal.'